In [ ]:
# =========================================================
# PLANTILLA BASE PARA CREAR MAPAS INTERACTIVOS CON FOLIUM
# =========================================================
# Autor: [Tu nombre o equipo]
# Descripción: 
#   Esta plantilla permite crear un mapa interactivo con:
#   - Puntos geográficos cargados desde un Excel (latitud/longitud)
#   - Una capa poligonal (por ejemplo, zonas FAO) cargada desde un shapefile
#   - Fondo base de CartoDB Positron
#   - Visualización directa en Jupyter Notebook
# =========================================================


# === 1. Carga de librerías ===
import pandas as pd
import geopandas as gpd
import folium
from folium.plugins import HeatMap
import os


# === 2. Definición de rutas de archivos ===
# Modifica las rutas según tu entorno de trabajo
ruta_excel = r"RUTA\AL\ARCHIVO\datos_puntos.xlsx"       # Archivo Excel con columnas Latitud y Longitud
ruta_shapefile = r"RUTA\AL\ARCHIVO\capa_poligonos.shp"  # Capa poligonal (ej. zonas FAO)
ruta_salida = r"RUTA\DE\SALIDA\mapa_interactivo.html"   # Archivo HTML de salida (opcional)


# === 3. Carga de datos ===
# Leemos los datos del Excel con pandas
df = pd.read_excel(ruta_excel)

# Leemos la capa poligonal con geopandas
capa = gpd.read_file(ruta_shapefile)


# === 4. Creación del GeoDataFrame con los puntos ===
# Asegúrate de que el Excel tenga las columnas "Latitud" y "Longitud"
gdf = gpd.GeoDataFrame(
    df,
    geometry=gpd.points_from_xy(df["Longitud"], df["Latitud"]),
    crs="EPSG:4326"   # Sistema de referencia geográfico (WGS84)
)


# === 5. Creación del mapa base ===
# Calculamos el centro del mapa a partir del promedio de coordenadas
centro = [gdf.geometry.y.mean(), gdf.geometry.x.mean()]

# Creamos el mapa base
m = folium.Map(
    location=centro,
    zoom_start=6,
    tiles='CartoDB positron',
    name='CartoDB Positron'
)


# === 6. Dibujado de puntos en el mapa ===
for _, row in gdf.iterrows():
    folium.CircleMarker(
        location=[row.geometry.y, row.geometry.x],
        radius=2,
        color="blue",
        fill=True,
        fill_color="blue",
        fill_opacity=0.6
    ).add_to(m)


# === 7. Asegurar CRS compatible y añadir la capa poligonal ===
if capa.crs is None or capa.crs.to_string().upper() != "EPSG:4326":
    capa = capa.to_crs(epsg=4326)

folium.GeoJson(
    capa,
    name="Capa poligonal",
    style_function=lambda x: {
        "color": "red",
        "fillColor": "transparent",
        "weight": 1.5
    }
).add_to(m)


# === 8. Visualizar el mapa en el notebook ===
m


# === 9. (Opcional) Guardar el mapa en HTML ===
# m.save(ruta_salida)
# print(f"✅ Mapa guardado correctamente en: {ruta_salida}")
